# Sinh prompt B1 từ cache B2

Có graph 40/80; không mã/nhãn demo; giữ câu dẫn demo trống như B0.

Notebook chỉ đọc cache và sinh prompt, không gọi LLM, không chạy Joern/CodeT5 và không viết notebook inference.
Mặc định đủ Devign/REVEAL validation/test, theo đúng tập ID và demo B2 đã có. Không âm thầm chọn lại demo bằng kho donor mới: các cặp flip/rep cần cùng tham chiếu B2.


In [1]:
from pathlib import Path
import sys,subprocess,json
INSTALL_DEPENDENCIES=False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','numpy','transformers==4.57.1','sentencepiece'],check=True)
PROJECT_ROOT_OVERRIDE=None  # Folder GRACE-summary nếu kernel mở ngoài repo.
candidates=[Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else [Path.cwd(),*Path.cwd().parents]
ROOT=next((p.resolve() for p in candidates if (p/'tier1_prompt_tools/prepare_cached.py').is_file()),None)
if ROOT is None: raise FileNotFoundError('Mở notebook trong repo hoặc đặt PROJECT_ROOT_OVERRIDE')
ARM='B1'
ARM_ROOT=ROOT/'03_Arm_B1_Graph_Only'
BASELINE=ROOT/'01_Author_Baseline_Unified'
REFERENCE_ROOT=BASELINE/'output/local_full/bundles_recovered_joern_final_20261031'
FROZEN_ROOT=BASELINE/'output/research_freeze_20261101'
OUT_ROOT=ARM_ROOT/'output/bundles'
RUNS=[('devign','validation'),('reveal','validation'),('devign','test'),('reveal','test')]
sys.path.insert(0,str(ROOT/'tier1_prompt_tools'))
import importlib, prepare_cached
importlib.reload(prepare_cached)  # Nạp bản tối ưu mới kể cả khi kernel đã import module trước đó.
from prepare_cached import prepare,cache_paths,render_paired,convert_graph,rows
print('Arm:',ARM)
print('B2 reference:',REFERENCE_ROOT)
print('Output:',OUT_ROOT)


Arm: B1
B2 reference: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/bundles_recovered_joern_final_20261031
Output: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/03_Arm_B1_Graph_Only/output/bundles


## Kiểm tra cache đã có

Specs B2 đã chứa CPG Joern và demo chọn bằng CodeT5. Tái sử dụng trực tiếp chúng là tầng cache cao nhất; không cần load encoder, chạy retrieval hay copy các file vectors.npy sang folder arm. Manifest/hash B2 và prompt từng mẫu sẽ được kiểm tra. Nếu B2 thay đổi, phải chọn thư mục tham chiếu đúng, không trộn hai phiên bản.


In [2]:
for dataset,split in RUNS:
    files,manifest=cache_paths(dataset,split,REFERENCE_ROOT)
    print(dataset,split,'B2 rows:',manifest['count'],'tokenizer:',manifest['tokenizer_revision'])
for dataset in sorted({d for d,s in RUNS}):
    for split in ('train','validation','test'):
        cache=BASELINE/'output/local_full'/dataset/'embedding_cache'/split
        m=json.loads((cache/'manifest.json').read_text(encoding='utf-8'))
        assert m['model']=='Salesforce/codet5-base@02cd2d31bb7c6d0e4d91156167b2de044989c733'
        assert m['completed']==len(m['rows']) and (cache/'vectors.npy').is_file(),'CodeT5 cache incomplete'
        print('CodeT5 cache reused via B2 demo selection:',dataset,split,m['completed'])


devign validation B2 rows: 2732 tokenizer: b2cff646eb4bb1d68355c01b18ae02e7cf42d120
reveal validation B2 rows: 2273 tokenizer: b2cff646eb4bb1d68355c01b18ae02e7cf42d120
devign test B2 rows: 2732 tokenizer: b2cff646eb4bb1d68355c01b18ae02e7cf42d120
reveal test B2 rows: 2274 tokenizer: b2cff646eb4bb1d68355c01b18ae02e7cf42d120
CodeT5 cache reused via B2 demo selection: devign train 21854
CodeT5 cache reused via B2 demo selection: devign validation 2732
CodeT5 cache reused via B2 demo selection: devign test 2732
CodeT5 cache reused via B2 demo selection: reveal train 18187
CodeT5 cache reused via B2 demo selection: reveal validation 2273
CodeT5 cache reused via B2 demo selection: reveal test 2274


In [3]:
from transformers import AutoTokenizer
first_files,first_manifest=cache_paths(*RUNS[0],REFERENCE_ROOT)
tokenizer=AutoTokenizer.from_pretrained(first_manifest['tokenizer'],revision=first_manifest['tokenizer_revision'],trust_remote_code=False)
assert tokenizer.chat_template
from grace_diag.token_budget import TokenCounter
counter=TokenCounter(None); counter.tok=tokenizer; counter.approx=False
spec=next(rows(first_files['specs']))
reference=next(r for r in rows(first_files['prompts']) if str(r['id'])==str(spec['id']))
preview,evidence=render_paired(ARM,spec,reference['prompt'],counter,reference=reference)
print(preview)
print(json.dumps(evidence,ensure_ascii=False,indent=2))
if ARM=='B2-rep': assert preview==reference['prompt']


/mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You are now an excellent programmer. You are conducting a function vulnerability detection task for C/C++ language.

static void filter_mirror_setup(NetFilterState *nf, Error **errp)
{
    MirrorState *s = FILTER_MIRROR(nf);
    Chardev *chr;
    chr = qemu_chr_find(s->outdev);
    if (chr == NULL) {
        error_set(errp, ERROR_CLASS_DEVICE_NOT_FOUND,
                  "Device '%s' not found", s->outdev);
    qemu_chr_fe_init(&s->chr_out, chr, errp);

In the above code snippet, check for potential security vulnerabilities and answer with a single digit: 1 if Vulnerable, 0 if Non-vulnerable.

The node information of the code snippet is as follows:
Node 25769803776: (BLOCK) '{     MirrorState *s = FILTER_MIRROR(nf)...'
Node 30064771072: (CALL) 's = FILTER_MIRROR(nf)'
Node 30064771073: (CALL) 'FILTER_MIRROR(nf)'
Node 30064771074: (CALL) 'chr = qemu_chr_find(s->outdev)'
Node 30064771075: (CALL) 'qemu_chr_find(s->outdev)'
Node 30064771076: (CALL) 's->outdev'
Node 55834574848: (FIELD_IDENT

In [4]:
# Nạp lại helper trước khi chạy cell này để tránh dùng bản chậm đã nằm trong kernel.
import importlib, prepare_cached
importlib.reload(prepare_cached)
prepare=prepare_cached.prepare
bundles,archive=prepare(ARM,REFERENCE_ROOT,OUT_ROOT,RUNS,tokenizer,
                      frozen_root=FROZEN_ROOT if FROZEN_ROOT.is_dir() else None)
print('Archive:',archive)
for path in bundles: print(path)


[B1 devign/validation] checking B2 source hashes...
[B1 devign/validation] verified completed bundle, reuse 2732 prompts
[B1 reveal/validation] checking B2 source hashes...
[B1 reveal/validation] 100/2273 | 29.7 samples/s | ETA 73s
[B1 reveal/validation] 200/2273 | 32.6 samples/s | ETA 64s
[B1 reveal/validation] 300/2273 | 39.7 samples/s | ETA 50s
[B1 reveal/validation] 400/2273 | 44.9 samples/s | ETA 42s
[B1 reveal/validation] 500/2273 | 45.1 samples/s | ETA 39s
[B1 reveal/validation] 600/2273 | 46.1 samples/s | ETA 36s
[B1 reveal/validation] 700/2273 | 47.5 samples/s | ETA 33s
[B1 reveal/validation] 800/2273 | 49.4 samples/s | ETA 30s
[B1 reveal/validation] 900/2273 | 50.6 samples/s | ETA 27s
[B1 reveal/validation] 1000/2273 | 51.4 samples/s | ETA 25s
[B1 reveal/validation] 1100/2273 | 50.8 samples/s | ETA 23s
[B1 reveal/validation] 1200/2273 | 52.8 samples/s | ETA 20s
[B1 reveal/validation] 1300/2273 | 51.7 samples/s | ETA 19s
[B1 reveal/validation] 1400/2273 | 52.2 samples/s | ETA 

## Đầu ra và đối chiếu

Mỗi split có prompts.jsonl, pairing_audit.jsonl, prompt_preview.txt, bundle_manifest.json và ZIP. Manifest ghi hash B2 gốc, số prompt giống B2, số graph thiếu và số demo B2 không thuộc kho donor mới. Đây là dữ liệu **ghép cặp với B2 đã có**, không tự chứng nhận tuân thủ snapshot donor mới. Nếu cần B2 xác nhận trên kho mới, phải tạo B2 mới rồi dùng cùng cache đó cho cả flip/rep.

B2-rep bắt buộc giống prompt B2 100%; các response/score B2 không được tái sử dụng khi chạy inference sau này. Với B1 không có demo trong prompt; nhãn thật của demo tham chiếu chỉ nằm ở pairing_audit, không đi vào renderer/prompt. Notebook sẽ dừng nếu hash nguồn, ID, nhãn target hoặc renderer B2 không khớp.
